<a href="https://colab.research.google.com/github/santidrelew/tp-mpg/blob/main/notebooks/02_extraccion_muestra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **2 - Extracción de la muestra**

In [13]:
try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    !git clone https://github.com/santidrelew/tp-mpg.git
    %cd tp-mpg

Cloning into 'tp-mpg'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 51 (delta 20), reused 8 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 4.54 MiB | 10.51 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/tp-mpg/tp-mpg/tp-mpg


In [14]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats

In [15]:
# Cargamos los datasets
PROJECT_DIR = Path().resolve() if EN_COLAB else Path().resolve().parent
DATA_BASE_DIR = PROJECT_DIR / "data" / "base"

df = pd.read_csv(DATA_BASE_DIR / "milan_airbnb_completo.csv")

Vamos a hacer un muestreo estratificado tomando como estrato la variable neighbourhood_cleansed. Esto es porque las tarifas y amenidades difieren sustancialmente entre el centro historico y las comunas perifericas. Un muestreo puramente aleatorio corria el riesgo de subestimar zonas de menor densidad de oferta (como la comuna 7).

In [16]:
N_MUESTRA = 750
fraccion = N_MUESTRA / len(df)

df_muestra = df.groupby("neighbourhood_cleansed", group_keys=False).apply(
    lambda x: x.sample(frac=fraccion, random_state=42)
)

# Redondeamos
if len(df_muestra) > N_MUESTRA:
  df_muestra = df_muestra.sample(n=N_MUESTRA, random_state=42)

/tmp/ipykernel_2459/3148027700.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_muestra = df.groupby("neighbourhood_cleansed", group_keys=False).apply(


In [17]:
TARGET = "daily_price"

media_pop = df[TARGET].mean()
var_pop = df[TARGET].var()

media_mue = df_muestra[TARGET].mean()
var_mue = df_muestra[TARGET].var()

tabla_validacion = pd.DataFrame(
    {
        "Metrica": [
            "Media (€)",
            "Varianza",
            "Desvio Estándar (€)",
            "Mediana (€)",
        ],
        "Poblacion": [
            round(media_pop, 2),
            round(var_pop, 2),
            round(df[TARGET].std(), 2),
            round(df[TARGET].median(), 2),
        ],
        "Muestra (n=750)": [
            round(media_mue, 2),
            round(var_mue, 2),
            round(df_muestra[TARGET].std(), 2),
            round(df_muestra[TARGET].median(), 2),
        ],
        "Diferencia Relativa (%)": [
            round(abs(media_pop - media_mue) / media_pop * 100, 2),
            round(abs(var_pop - var_mue) / var_pop * 100, 2),
            round(
                abs(df[TARGET].std() - df_muestra[TARGET].std())
                / df[TARGET].std()
                * 100,
                2,
            ),
            round(
                abs(df[TARGET].median() - df_muestra[TARGET].median())
                / df[TARGET].median()
                * 100,
                2,
            ),
        ],
    }
)

display(tabla_validacion)

,Metrica,Poblacion,Muestra (n=750),Diferencia Relativa (%)
0,Media (€),103.74,99.55,4.03
1,Varianza,19031.98,8510.88,55.28
2,Desvio Estándar (€),137.96,92.25,33.13
3,Mediana (€),75.00,75.00,0.00


Como tanto la media como la mediana tienen una diferencia relativa de menos del 5% (la mediana, que es una medida robusta frente a la asimetria, es exactamente la misma), podemos decir que la muestra es representativa.

En cuanto a la varianza y el desvio estandar, la gran diferencia relativa no representa un sesgo de muestreo, sino que la poblacion total contiene valores extremos y atipicos en la cola superior (como las propiedades de $3.000$ €) que inflan la dispersion global. Al extraer una muestra aleatoria estratificada, la probabilidad de capturar dichos eventos aislados disminuye, lo que reduce la varianza muestral. A su vez, justifican el uso de log_daily_price para neutralizar los valores extremos en nuestra regresion.

In [18]:
# Guardamos la muestra
df_muestra.to_csv(DATA_BASE_DIR / "milan_airbnb_muestra.csv", index=False)